# Week 06 실습 · 파일 · CSV · 예외 처리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week06/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. `with open(...)`과 `pathlib`으로 텍스트 파일을 안전하게 쓰고 읽습니다.
2. `csv.DictReader` / `csv.DictWriter`로 CSV 파일을 읽고 씁니다. **CSV에서 읽은 값은 모두 문자열**입니다.
3. `try / except / else / finally`로 예상 가능한 오류를 처리하고, `raise`로 이유를 알립니다.
4. 오류가 섞인 성적 파일을 **검증 규칙**에 따라 정상 행과 오류 행으로 나누고 보고서를 저장합니다.
5. AI가 자주 쓰는 `except: pass`가 **버그를 숨기는** 위험을 확인합니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 0 | 데이터 준비 | 5분 |
| 1 | Level 1 텍스트 파일 | 15분 |
| 2 | Level 1 CSV 읽고 쓰기 | 20분 |
| 3 | Level 1 예외 처리 | 20분 |
| 4 | Level 2 바꾸기 | 10분 |
| 5 | Level 3 오류에 견고한 성적 파일 처리기 | 30분 |
| 6 | Level 4 AI 코드의 `except: pass` | 10분 |
| 7 | 정리 | 10분 |

---
## 0. 데이터 준비

저장소를 내려받아 실행하면 `datasets` 폴더의 파일을, Colab에서는 GitHub에서 파일을 내려받아 사용합니다.

In [ ]:
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    """datasets 폴더에 있으면 그 경로를, 없으면 GitHub에서 내려받은 경로를 돌려준다."""
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

RAW_PATH = get_data("students_raw.csv")
print("사용할 파일:", RAW_PATH)

---
## 1. Level 1 따라하기: 텍스트 파일

| 모드 | 의미 | 파일이 없으면 | 파일이 있으면 |
|---|---|---|---|
| `"r"` | 읽기 (기본값) | `FileNotFoundError` | 처음부터 읽음 |
| `"w"` | 쓰기 | 새로 만듦 | **내용을 지우고** 새로 씀 |
| `"a"` | 이어 쓰기 | 새로 만듦 | 끝에 덧붙임 |

`with` 블록을 쓰면 블록이 끝날 때 파일이 **자동으로 닫힙니다.** 한글은 `encoding="utf-8"`을 꼭 지정합니다.

In [ ]:
with open("memo.txt", "w", encoding="utf-8") as f:
    f.write("1주차: Colab 시작\n")
    f.write("2주차: 문자열\n")

with open("memo.txt", "a", encoding="utf-8") as f:
    f.write("3주차: 조건문과 반복문\n")

with open("memo.txt", encoding="utf-8") as f:
    content = f.read()
print(content)

In [ ]:
# 한 줄씩 읽기: 각 줄 끝에는 줄바꿈 문자 '\n'이 붙어 있다
with open("memo.txt", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        print(line_no, repr(line), "→", line.strip())

### `pathlib.Path`: 경로를 객체로 다루기

In [ ]:
path = Path("memo.txt")
print("존재?", path.exists(), "/ 크기", path.stat().st_size, "바이트 / 확장자", path.suffix)

text = path.read_text(encoding="utf-8")
lines = text.splitlines()
print(lines)

Path("out").mkdir(exist_ok=True)
(Path("out") / "summary.txt").write_text(f"총 {len(lines)}줄\n", encoding="utf-8")
print(list(Path("out").iterdir()))

---
## 2. Level 1 따라하기: CSV 읽고 쓰기

CSV(Comma-Separated Values)는 쉼표로 열을 나눈 표 데이터입니다. 첫 줄은 보통 **열 이름(헤더)** 입니다.

In [ ]:
with open(RAW_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i == 5:
            break

### `csv.DictReader`: 각 행을 딕셔너리로

⚠️ **CSV에서 읽은 값은 숫자처럼 보여도 모두 문자열입니다.** 빈 칸은 빈 문자열 `""`입니다.

In [ ]:
import csv

with open(RAW_PATH, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    print("열 이름:", reader.fieldnames)
    rows = list(reader)

print(len(rows), "행")
print(rows[0])
print("midterm 자료형:", type(rows[0]["midterm"]))
print("3번째 학생 과제:", repr(rows[2]["assignment"]))

In [ ]:
# 문자열 그대로 더하면?
print(rows[0]["midterm"] + rows[0]["final"])          # '7885' ← 이어 붙여짐
print(int(rows[0]["midterm"]) + int(rows[0]["final"]))  # 163

### `csv.DictWriter`: 딕셔너리 리스트를 CSV로

In [ ]:
summary = [
    {"name": "김민준", "total": 82.2, "grade": "B"},
    {"name": "이서연", "total": 89.6, "grade": "B"},
]

with open("out/summary.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "total", "grade"])
    writer.writeheader()
    writer.writerows(summary)

print(Path("out/summary.csv").read_text(encoding="utf-8"))

> 💡 `newline=""`은 Windows에서 빈 줄이 끼어드는 문제를 막습니다. CSV를 열 때는 습관적으로 붙입니다.
> 엑셀에서 한글이 깨지면 `encoding="utf-8-sig"`로 저장합니다.

---
## 3. Level 1 따라하기: 예외 처리

```python
try:
    오류가 날 수 있는 코드
except 특정오류 as e:
    그 오류가 났을 때
else:
    오류가 없었을 때
finally:
    오류와 상관없이 항상
```

In [ ]:
def to_score(text):
    try:
        value = int(text.strip())
    except ValueError:
        print(f"  숫자가 아님: {text!r}")
        return None
    else:
        print(f"  변환 성공: {value}")
        return value
    finally:
        print("  (변환 시도 끝)")

for t in ["85", " 92 ", "abc", ""]:
    print(t, "→", to_score(t))

### 여러 종류의 오류를 따로 처리하기

In [ ]:
def average_of(path, column):
    try:
        with open(path, encoding="utf-8", newline="") as f:
            values = [int(r[column]) for r in csv.DictReader(f)]
        return sum(values) / len(values)
    except FileNotFoundError:
        return "파일 없음"
    except KeyError:
        return f"'{column}' 열 없음"
    except ValueError as e:
        return f"숫자 변환 실패 ({e})"
    except ZeroDivisionError:
        return "데이터 없음"

print(average_of("없는파일.csv", "midterm"))
print(average_of(RAW_PATH, "점수"))
print(average_of(RAW_PATH, "midterm"))
print(average_of(RAW_PATH, "year"))

### `raise`: 이유를 담아 오류 발생시키기

검증 함수는 문제를 발견하면 **무엇이 왜 잘못됐는지** 메시지에 담아 `raise`합니다. 부르는 쪽에서 `except ... as e`로 받아 기록합니다.

In [ ]:
def parse_score(text, field):
    """0~100 정수 점수로 바꾼다. 실패하면 이유를 담은 ValueError."""
    text = text.strip()
    if text == "":
        raise ValueError(f"{field}: 값이 비어 있음")
    try:
        value = int(text)
    except ValueError:
        raise ValueError(f"{field}: 숫자가 아님 ({text!r})") from None
    if not 0 <= value <= 100:
        raise ValueError(f"{field}: 0~100 범위 밖 ({value})")
    return value

for t in ["85", "", "abc", "120"]:
    try:
        print(parse_score(t, "midterm"))
    except ValueError as e:
        print("오류 →", e)

---
## 4. Level 2 바꾸기

### ✏️ 직접 해보기 1

`to_score`를 고쳐서 **0~100 범위 밖**이면 `None`을 돌려주도록 해 보세요. `"120"`, `"-5"`로 확인합니다.

In [ ]:
def to_score_v2(text):
    try:
        value = int(text.strip())
    except ValueError:
        return None
    # ← 여기에 범위 검사를 추가해 보세요
    return value

for t in ["85", "120", "-5", "abc"]:
    print(t, "→", to_score_v2(t))

### ✏️ 직접 해보기 2

`memo.txt`에 4주차, 5주차 내용을 **이어 쓰기** 모드로 추가하고, 전체 줄 수를 출력해 보세요. `"w"` 모드로 바꾸면 어떻게 되는지도 확인합니다.

In [ ]:
with open("memo.txt", "a", encoding="utf-8") as f:    # ← 모드를 바꿔 보세요
    f.write("4주차: 자료구조\n")

print(len(Path("memo.txt").read_text(encoding="utf-8").splitlines()), "줄")

---
## 5. Level 3 적용하기: 오류에 견고한 성적 파일 처리기

**검증 규칙**

| 열 | 규칙 |
|---|---|
| `student_id` | 8자리 숫자, **중복 불가** |
| `name` | 비어 있으면 안 됨 |
| `year` | 1~4 정수 |
| `midterm`, `final` | 필수, 0~100 정수 (앞뒤 공백 허용) |
| `assignment` | 선택(빈 칸이면 `None`), 있으면 0~100 정수 |
| `attendance` | 0~16 정수 |

**출력**: 정상 행은 `out/students_clean.csv`, 오류는 `out/error_report.csv` (줄 번호, 학번, 이유)

### 5-1. 한 행 검증 함수

In [ ]:
def parse_int(text, field, low, high, required=True):
    """문자열을 low~high 정수로 바꾼다. 선택 항목이 비어 있으면 None."""
    text = text.strip()
    if text == "":
        if required:
            raise ValueError(f"{field}: 값이 비어 있음")
        return None
    try:
        value = int(text)
    except ValueError:
        raise ValueError(f"{field}: 숫자가 아님 ({text!r})") from None
    if not low <= value <= high:
        raise ValueError(f"{field}: {low}~{high} 범위 밖 ({value})")
    return value


def validate_row(row):
    """CSV 한 행(dict)을 검증해 변환된 레코드를 돌려준다. 문제가 있으면 ValueError."""
    sid = row["student_id"].strip()
    if not (len(sid) == 8 and sid.isdigit()):
        raise ValueError(f"student_id: 8자리 숫자가 아님 ({sid!r})")
    name = row["name"].strip()
    if not name:
        raise ValueError("name: 값이 비어 있음")
    return {
        "student_id": sid,
        "name": name,
        "department": row["department"].strip(),
        "year": parse_int(row["year"], "year", 1, 4),
        "midterm": parse_int(row["midterm"], "midterm", 0, 100),
        "final": parse_int(row["final"], "final", 0, 100),
        "assignment": parse_int(row["assignment"], "assignment", 0, 100, required=False),
        "attendance": parse_int(row["attendance"], "attendance", 0, 16),
    }

print(validate_row(rows[0]))
try:
    validate_row(rows[4])
except ValueError as e:
    print("오류:", e)

### 5-2. 파일 전체 처리

줄 번호는 **헤더가 1번 줄**이므로 데이터 첫 행이 2번 줄입니다. 오류 보고서에 줄 번호가 있어야 원본 파일을 고치기 쉽습니다.

In [ ]:
def process_file(path):
    valid, errors = [], []
    seen_ids = set()
    with open(path, encoding="utf-8", newline="") as f:
        for line_no, row in enumerate(csv.DictReader(f), start=2):
            try:
                record = validate_row(row)
                if record["student_id"] in seen_ids:
                    raise ValueError(f"student_id: 중복 ({record['student_id']})")
            except ValueError as e:
                errors.append({"line": line_no, "student_id": row["student_id"], "reason": str(e)})
            else:
                seen_ids.add(record["student_id"])
                valid.append(record)
    return valid, errors


valid, errors = process_file(RAW_PATH)
print(f"정상 {len(valid)}행, 오류 {len(errors)}행")
for e in errors:
    print(f"  {e['line']:>2}번 줄 [{e['student_id']}] {e['reason']}")

### 5-3. 결과 저장과 검증

In [ ]:
def write_csv(path, records, fieldnames):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


write_csv("out/students_clean.csv", valid, list(valid[0].keys()))
write_csv("out/error_report.csv", errors, ["line", "student_id", "reason"])

print(Path("out/error_report.csv").read_text(encoding="utf-8"))

In [ ]:
# 검증: 행 수가 맞는가? 정상 파일을 다시 읽으면 모두 통과하는가?
with open(RAW_PATH, encoding="utf-8", newline="") as f:
    total_rows = len(list(csv.DictReader(f)))
assert len(valid) + len(errors) == total_rows

re_valid, re_errors = process_file("out/students_clean.csv")
assert len(re_valid) == len(valid) and re_errors == []
assert len({r["student_id"] for r in valid}) == len(valid)
print("검증 통과 ✅  (정상 행만 담은 파일은 다시 검증해도 오류 0건)")

> 💡 정상 파일을 다시 읽을 때 `assignment`가 `None`이었던 칸은 CSV에 빈 칸으로 저장되므로, 다시 읽어도 `None`이 됩니다.

---
## 6. Level 4 AI와 함께: `except: pass`는 버그를 숨긴다

"CSV에서 midterm 평균을 구하는데 오류 나는 행은 무시해 줘"라고 요청해 받은 코드입니다. **오타가 하나** 숨어 있습니다.

In [ ]:
def midterm_average_ai(path):
    total, count = 0, 0
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            try:
                score = int(row["midterm"])
                if 0 <= score <= 100:
                    total += score
                    conut += 1             # 오타! (count)
            except:
                pass
    return total / count if count else 0

print(midterm_average_ai(RAW_PATH))

결과가 `0`입니다. 오류 메시지는 하나도 없습니다.

- `conut += 1`에서 `UnboundLocalError`(이름 오류의 일종)가 **모든 정상 행에서** 발생했지만 `except: pass`가 조용히 삼켰습니다.
- 그래서 `count`는 끝까지 0이고, 함수는 "데이터가 없다"는 뜻의 0을 돌려줍니다.
- 숫자가 아닌 `"abc"` 행(정말 건너뛰고 싶었던 오류)과 **오타라는 프로그래머의 실수**가 똑같이 취급되어, 무엇이 잘못됐는지 알 수 없습니다.

### 🛠️ 고치는 원칙

1. **예상한 오류만** 잡는다: `except ValueError:`
2. 무시하더라도 **몇 건을 왜 건너뛰었는지 기록**한다
3. 예상하지 못한 오류(오타로 인한 `KeyError` 등)는 **그대로 터지게** 둔다 → 바로 발견된다

In [ ]:
def midterm_average(path):
    total, count, skipped = 0, 0, []
    with open(path, encoding="utf-8", newline="") as f:
        for line_no, row in enumerate(csv.DictReader(f), start=2):
            try:
                value = parse_int(row["midterm"], "midterm", 0, 100)
            except ValueError as e:
                skipped.append((line_no, str(e)))
                continue
            total += value
            count += 1
    return round(total / count, 1), skipped

avg, skipped = midterm_average(RAW_PATH)
print("평균:", avg)
print("건너뛴 행:", skipped)

### 📝 AI에게 예외 처리를 요청할 때

```text
- bare except(except:)나 except Exception: pass 는 쓰지 마세요.
- int 변환 실패처럼 예상 가능한 ValueError만 잡고, 건너뛴 행의 줄 번호와 이유를 리스트로 함께 반환하세요.
- 파일이 없으면 FileNotFoundError가 그대로 발생하게 두세요.
```

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 파일 | `with open(path, mode, encoding="utf-8")`, 모드 `r` `w` `a` |
| pathlib | `Path.exists()`, `read_text`, `write_text`, `mkdir(exist_ok=True)` |
| CSV | `DictReader`/`DictWriter`, `newline=""`, **값은 모두 문자열** |
| 예외 처리 | `try / except 특정오류 as e / else / finally` |
| raise | 무엇이 왜 잘못됐는지 메시지에 담기 |
| 견고한 처리 | 행 단위 검증 → 정상/오류 분리 → 줄 번호 보고서 → 결과 재검증 |
| AI 검증 | `except: pass` 금지, 예상한 오류만, 건너뛴 이유 기록 |

### 자기 점검

- [ ] `"w"`와 `"a"` 모드의 차이를 설명할 수 있다.
- [ ] CSV에서 읽은 `"78"` + `"85"`가 `"7885"`가 되는 이유를 안다.
- [ ] `except:`가 위험한 이유를 예를 들어 설명할 수 있다.
- [ ] 오류 보고서에 줄 번호를 남기는 이유를 말할 수 있다.

### 다음 주 예고

**Week 07 · 클래스와 객체지향 설계**: 상태(데이터)와 동작(함수)을 함께 관리하는 도서 대출 관리 클래스를 만듭니다.